In [2]:
import torch, numpy as np
from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import r2_score, mean_squared_error

AMINO_ACIDS = "ACDEFGHIKLMNPQRSTVWY"
stoi = {aa: i for i, aa in enumerate(AMINO_ACIDS)}

def encode(seq: str) -> torch.Tensor:
    return torch.tensor([stoi[c] for c in seq], dtype=torch.long)

# "YYDPETGQWY": 9080,
seq_to_val = {
    "YYDPETGTWY": 12540, "YYNPETGTWY": 7860, "YYAPETGTWY": 7794,
    "YYDPETGTWE": 3178,  "YYRPETGTWY": 5899,
    "YYDPETGRWY": 10670, "YYCPETGTWY": 10682, "YYDPETGTWQ": 5392,
    "YYDPETGTWG": 4142, "YYMPETGTWY": 12290, "YYDPETGTWV": 7590,
    "YYEPETGTWY": 9548, "YYDPETGYWY": 8645, "YYDPETGTWR": 5799,
    "YYDPETGVWY": 13050, "YYDPETGTWA": 5520, "YYDPETGDWY": 7538,
    "YYDPETGGWY": 4623, "AYDPETGTWY": 12425, "EYDPETGTWY": 9754,
    "QYDPETGTWY": 9935, "RYDPETGTWY": 9961, "YYDCETGTWY": 6460,
    "YYDDETGTWY": 4244, "YYDMETGTWY": 8955, "YYDRETGTWY": 5388,
}

class SeqDataset(Dataset):
    def __init__(self, d):
        xs, ys = [], []
        for s,v in d.items():
            xs.append(encode(s))
            ys.append(np.log(float(v)))
        y = torch.tensor(ys, dtype=torch.float32)
        self.y_mean = y.mean().item()
        self.y_std = y.std(unbiased=False).item()
        self.X = torch.stack(xs)
        self.y = (y - self.y_mean) / (self.y_std + 1e-8)
    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i], self.y[i]

class PositionalEmbedding(nn.Module):
    def __init__(self, seq_len, d_model):
        super().__init__()
        self.pe = nn.Parameter(torch.zeros(1, seq_len, d_model))
        nn.init.normal_(self.pe, std=0.02)
    def forward(self, x): return x + self.pe

class SeqRegressor(nn.Module):
    def __init__(self, vocab_size, seq_len=10, d=48, nhead=4, nlayers=2, p=0.2):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, d)
        self.pos = PositionalEmbedding(seq_len, d)
        enc = nn.TransformerEncoderLayer(d_model=d, nhead=nhead, dim_feedforward=4*d,
                                         dropout=p, batch_first=True, activation="gelu")
        self.tr = nn.TransformerEncoder(enc, num_layers=nlayers)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.head = nn.Sequential(nn.LayerNorm(d), nn.Linear(d, 64), nn.GELU(),
                                  nn.Dropout(p), nn.Linear(64, 1))
    def forward(self, x):
        z = self.emb(x)
        z = self.pos(z)
        z = self.tr(z)
        z = z.transpose(1,2)
        z = self.pool(z).squeeze(-1)
        return self.head(z).squeeze(-1)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ds = SeqDataset(seq_to_val)
X, y = ds.X.to(device), ds.y.to(device)
model = SeqRegressor(vocab_size=len(AMINO_ACIDS)).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=5e-4)
loss_fn = nn.MSELoss()

for epoch in range(400):
    model.train()
    idx = torch.randperm(len(ds))
    total = 0.0
    for i in range(0, len(ds), 8):
        b = idx[i:i+8]
        pred = model(X[b])
        loss = loss_fn(pred, y[b])
        opt.zero_grad(); loss.backward(); opt.step()
        total += loss.item() * len(b)
    if (epoch+1) % 50 == 0:
        print(f"epoch {epoch+1}: train_mse={total/len(ds):.4f}")

model.eval()
with torch.no_grad():
    pred = model(encode("YYDPETGQWY").unsqueeze(0).to(device))
    val = pred.item()*ds.y_std + ds.y_mean
    print("YYDPETGQWY pred:", float(np.exp(val)))

loo = LeaveOneOut()
y_true, y_pred = [], []
with torch.no_grad():
    for i, (train_idx, test_idx) in enumerate(loo.split(X.cpu())):
        m = SeqRegressor(vocab_size=len(AMINO_ACIDS)).to(device)
        o = torch.optim.AdamW(m.parameters(), lr=2e-3, weight_decay=5e-4)
        for epoch in range(250):
            m.train()
            bi = torch.tensor(train_idx, device=device)
            pred = m(X[bi]); loss = loss_fn(pred, y[bi])
            o.zero_grad(); loss.backward(); o.step()
        m.eval()
        yi = y[test_idx].cpu().numpy()
        pi = m(X[test_idx]).cpu().numpy()
        y_true.append(yi[0]); y_pred.append(pi[0])
y_true = np.array(y_true); y_pred = np.array(y_pred)
r2 = r2_score(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred)) * ds.y_std
print("LOO R2:", r2, "LOO RMSE (in log-space units→multiplier):", rmse)


epoch 50: train_mse=0.5874
epoch 100: train_mse=0.0886
epoch 150: train_mse=0.0603
epoch 200: train_mse=0.0940
epoch 250: train_mse=0.0387
epoch 300: train_mse=0.0433
epoch 350: train_mse=0.0355
epoch 400: train_mse=0.0238
YYDPETGQWY pred: 10656.676447422471


RuntimeError: element 0 of tensors does not require grad and does not have a grad_fn

In [ ]:
# -*- coding: utf-8 -*-
"""
MFPT regressor (simple) using fixed d-columns only, user-selected mutants, and no path existence checks.
- Uses only these columns: d03,d04,...,d69 (see D_COLS)
- Builds WT–Mut delta channels [Mut, WT, Mut−WT, |Mut−WT|]
- Tiny 1D-CNN over windows predicts log(MFPT)
- Validates at mutant level (avg window preds)

Set ACTIVE_MUTANTS to the exact subset you have trajectories for (MUST include WT_KEY).
Files are assumed at: ../data/<protein>/output/run_001/COLVAR_001 (extend RUN_IDS/COLVAR_IDS if needed).
If anything is missing, it will raise (by design).
"""

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from pathlib import Path
from torch.utils.data import Dataset, DataLoader

# ----------------------------
# 0) USER INPUTS
# ----------------------------

MFPT_US = {
    "YYAPETGTWY": 0.1507, "YYCPETGTWY": 1.768, "YYMPETGTWY": 0.9062,
    "YYNPETGTWY": 0.05355, "YYRPETGTWY": 0.1758, "YYEPETGTWY": 0.723,
    "YYDPETGTWE": 0.01073, "YYDPETGTWG": 0.001776, "YYDPETGTWQ": 0.01118,
    "YYDPETGTWR": 0.03271, "YYDPETGTWV": 0.01123, "YYDPETGTWA": 0.009195,
    "YYDPETGVWY": 1.188, "YYDPETGQWY": 3.275, "YYDPETGRWY": 3.098,
    "YYDPETGYWY": 0.18, "YYDPETGGWY": 0.00353, "YYDPETGDWY": 0.3838,
    "chignolin": 0.08167, "AYDPETGTWY": 0.04986, "YYDCETGTWY": 0.292,
    "RYDPETGTWY": 0.07572, "QYDPETGTWY": 0.06231, "EYDPETGTWY": 0.6906,
    "YYDMETGTWY": 1.144, "YYDDETGTWY": 0.04687, "YYDRETGTWY": 0.02634,
}

WT_KEY = "chignolin"

# Choose only the mutants/trajs you actually have (MUST include WT_KEY)
ACTIVE_MUTANTS = [
    # example subset — replace with your list
    "chignolin",
    "YYAPETGTWY", "YYNPETGTWY", "YYDPETGTWE", "YYDPETGTWV",
    "YYDPETGQWY", "YYDPETGRWY", "YYDPETGDWY",
]

BASE_ROOT = Path("../data")
OUTPUT_SUBDIR = "output"
RUN_IDS = (1,)     # or (1,2,3)
COLVAR_IDS = (1,)  # or (1,2)

D_COLS = [
    "d03","d04","d05","d06","d07","d08","d09",
    "d14","d15","d16","d17","d18","d19",
    "d25","d26","d27","d28","d29",
    "d36","d37","d38","d39",
    "d47","d48","d49",
    "d58","d59","d69",
]

WIN_LEN = 256
STRIDE = 128
BATCH = 16
EPOCHS = 40
LR = 1e-3
WEIGHT_DECAY = 1e-4
DROPOUT_P = 0.2

# ----------------------------
# 1) IO + PREP (no existence checks)
# ----------------------------

def build_paths_for_protein(protein: str) -> list[Path]:
    base = BASE_ROOT / protein / OUTPUT_SUBDIR
    return [base / f"run_{int(r):03d}" / f"COLVAR_{int(c):03d}" for r in RUN_IDS for c in COLVAR_IDS]

def get_fields(path: Path) -> list[str]:
    with open(path, "r") as f:
        for line in f:
            if line.startswith("#! FIELDS"):
                return line.replace("#! FIELDS", "").strip().split()
    raise ValueError(f"FIELDS not found in {path}")

def read_dcols(path: Path) -> np.ndarray:
    fields = get_fields(path)
    use = [c for c in D_COLS if c in fields]
    if len(use) != len(D_COLS):
        missing = [c for c in D_COLS if c not in fields]
        raise ValueError(f"Missing columns in {path}: {missing}")
    df = pd.read_csv(path, delim_whitespace=True, comment="#", header=None, names=fields, usecols=use)
    return df.to_numpy(dtype=np.float32)

def load_trajs_for_protein(protein: str) -> list[np.ndarray]:
    return [read_dcols(p) for p in build_paths_for_protein(protein)]

def fit_scaler(arrs: list[np.ndarray]) -> tuple[np.ndarray, np.ndarray]:
    X = np.concatenate(arrs, axis=0)
    m = X.mean(axis=0, keepdims=True)
    s = X.std(axis=0, keepdims=True) + 1e-8
    return m, s

def apply_scaler(A: np.ndarray, m: np.ndarray, s: np.ndarray) -> np.ndarray:
    return (A - m) / s

# ----------------------------
# 2) DATASET
# ----------------------------

def win_indices(T: int, L: int, stride: int, mode: str) -> list[int]:
    if T < L: return []
    if mode == "train":
        return list(range(0, T - L + 1, stride))
    step = max(1, L)
    return list(range(0, max(1, T - L + 1), step))

class WinDataset(Dataset):
    def __init__(self, keys: list[str], arrs: dict[str, list[np.ndarray]], wt_arrs: list[np.ndarray], y_log: dict[str, float],
                 L: int = WIN_LEN, stride: int = STRIDE, mode: str = "train", seed: int = 0):
        self.keys, self.arrs, self.wt_arrs = keys, arrs, wt_arrs
        self.y, self.L, self.stride, self.mode = y_log, L, stride, mode
        self.items: list[tuple[str, np.ndarray, int]] = []
        rng = np.random.default_rng(seed)
        for k in keys:
            for A in arrs[k]:
                T = A.shape[0]
                idxs = win_indices(T, L, stride, mode)
                if mode == "train" and len(idxs) == 0 and T >= L:
                    idxs = [rng.integers(0, T - L + 1)]
                for i in idxs:
                    self.items.append((k, A, i))
    def __len__(self):
        return len(self.items)
    def __getitem__(self, i: int):
        k, A, i0 = self.items[i]
        xm = A[i0:i0 + self.L]
        B = self.wt_arrs[np.random.randint(0, len(self.wt_arrs))]
        if B.shape[0] >= self.L:
            j = np.random.randint(0, B.shape[0] - self.L + 1)
            xw = B[j:j + self.L]
        else:
            pad = self.L - B.shape[0]
            xw = np.pad(B, ((0, pad), (0, 0)))
        x = np.concatenate([xm, xw, xm - xw, np.abs(xm - xw)], axis=1)
        return torch.from_numpy(x), torch.tensor(self.y[k], dtype=torch.float32), k

# ----------------------------
# 3) MODEL
# ----------------------------

class TinyCNN(nn.Module):
    def __init__(self, n_in: int, p: float = DROPOUT_P):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(n_in, 64, 5, padding=2), nn.GELU(), nn.Dropout(p),
            nn.Conv1d(64, 64, 5, padding=2), nn.GELU(),
            nn.Conv1d(64, 64, 5, padding=2), nn.GELU(),
        )
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.head = nn.Sequential(nn.LayerNorm(64), nn.Linear(64, 64), nn.GELU(), nn.Dropout(p), nn.Linear(64, 1))
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        z = self.net(x.transpose(1, 2))
        z = self.pool(z).squeeze(-1)
        return self.head(z).squeeze(-1)

# ----------------------------
# 4) TRAIN / EVAL
# ----------------------------

def train_one_epoch(model: nn.Module, loader: DataLoader, opt: torch.optim.Optimizer, loss_fn: nn.Module) -> float:
    model.train(); tot = 0.0; n = 0
    for xb, yb, _ in loader:
        pred = model(xb)
        loss = loss_fn(pred, yb)
        opt.zero_grad(); loss.backward(); opt.step()
        tot += loss.item() * len(xb); n += len(xb)
    return tot / max(1, n)

@torch.no_grad()
def eval_mutant_level(model: nn.Module, loader: DataLoader) -> tuple[float, float, dict[str, float]]:
    model.eval(); agg: dict[str, dict[str, list[float] | float]] = {}
    for xb, yb, ks in loader:
        pr = model(xb).cpu().numpy(); yt = yb.cpu().numpy()
        for p, t, k in zip(pr, yt, ks):
            d = agg.setdefault(k, {"p": [], "t": float(t)})
            d["p"].append(float(p))
    preds = np.array([np.mean(v["p"]) for v in agg.values()], dtype=float)
    trues = np.array([v["t"] for v in agg.values()], dtype=float)
    rmse = float(np.sqrt(((preds - trues) ** 2).mean()))
    def _rank(a):
        order = a.argsort(kind="mergesort"); ranks = np.empty_like(order, dtype=float); ranks[order] = np.arange(len(a)); return ranks
    r_pred, r_true = _rank(preds), _rank(trues)
    r_pred = (r_pred - r_pred.mean()) / (r_pred.std() + 1e-8)
    r_true = (r_true - r_true.mean()) / (r_true.std() + 1e-8)
    spearman = float(np.corrcoef(r_pred, r_true)[0, 1]) if len(preds) > 1 else 0.0
    pred_us = {k: float(np.exp(np.mean(v["p"]))) for k, v in agg.items()}
    return rmse, spearman, pred_us

# ----------------------------
# 5) WIRING (no checks; uses ACTIVE_MUTANTS only)
# ----------------------------

def prepare_arrays(mfpt: dict[str, float], active: list[str], wt_key: str = WT_KEY):
    assert wt_key in active, f"ACTIVE_MUTANTS must include WT '{wt_key}'"
    keys = list(active)
    raw = {k: load_trajs_for_protein(k) for k in keys}
    train_keys = [k for k in keys if k != wt_key]
    m, s = fit_scaler([A for k in train_keys for A in raw[k]])
    arrs = {k: [apply_scaler(A, m, s).astype(np.float32) for A in raw[k]] for k in keys}
    wt_arrs = arrs[wt_key]
    n_feats = arrs[train_keys[0]][0].shape[1]
    return keys, arrs, wt_arrs, n_feats


def run():
    keys, arrs, wt_arrs, n_feats = prepare_arrays(MFPT_US, ACTIVE_MUTANTS, WT_KEY)
    y_log = {k: float(np.log(MFPT_US[k])) for k in keys}
    train_keys = [k for k in keys if k != WT_KEY]

    ds_tr = WinDataset(train_keys, arrs, wt_arrs, y_log, L=WIN_LEN, stride=STRIDE, mode="train")
    ds_va = WinDataset(train_keys, arrs, wt_arrs, y_log, L=WIN_LEN, stride=STRIDE, mode="val")
    tr_loader = DataLoader(ds_tr, batch_size=BATCH, shuffle=True)
    va_loader = DataLoader(ds_va, batch_size=BATCH, shuffle=False)

    model = TinyCNN(n_in=4 * n_feats, p=DROPOUT_P)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    loss_fn = nn.MSELoss()

    best = (1e9, None)
    for epoch in range(1, EPOCHS + 1):
        tr_mse = train_one_epoch(model, tr_loader, opt, loss_fn)
        rmse_log, rho, _ = eval_mutant_level(model, va_loader)
        if rmse_log < best[0]:
            best = (rmse_log, {k: v.clone() if hasattr(v, "clone") else v for k, v in model.state_dict().items()})
        if epoch % 5 == 0:
            print(f"ep{epoch:03d} train_mse={tr_mse:.4f} val_rmse_log={rmse_log:.4f} spearman={rho:.3f}")

    model.load_state_dict(best[1])
    rmse_log, rho, pred_us = eval_mutant_level(model, va_loader)
    print("final val_rmse_log:", rmse_log, "spearman:", rho)
    print({k: round(v, 6) for k, v in pred_us.items()})


if __name__ == "__main__":
    run()
